In [1]:
!nvidia-smi

Tue Aug 11 07:11:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   49C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print(torch.cuda.get_device_properties(0).total_memory/1024**3)

2.11.0+cu128
NVIDIA L4
(8, 9)
22.0343017578125


In [3]:
%pip install -q \
    "vllm==0.19.0" \
    "pyngrok>=7,<8"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 124.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 97.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 2.1 MB/s eta 0:00:00:00:0100:01
 

In [4]:
import torch
from google.colab import userdata
import getpass
VLLM_API_KEY = getpass.getpass("VLLM_API_KEY: ")

assert VLLM_API_KEY, "Thiếu VLLM_API_KEY"
assert torch.cuda.is_available(), "Không tìm thấy CUDA"
assert torch.cuda.is_bf16_supported(), "GPU không hỗ trợ BF16"

gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3

print(f"GPU: {gpu.name}")
print(f"VRAM: {vram_gib:.1f} GiB")
print(f"CUDA: {torch.version.cuda}")

if vram_gib < 20:
    raise RuntimeError("Cần GPU khoảng 20 GiB trở lên; hãy lấy L4/A100.")


GPU: NVIDIA L4
VRAM: 22.0 GiB
CUDA: 12.8


In [5]:
import os
import torch

gpu = torch.cuda.get_device_properties(0)
gpu_name = gpu.name
gpu_memory_gib = gpu.total_memory / 1024**3
cpu_count = os.cpu_count() or 1

print(
    f"GPU={gpu_name}, "
    f"VRAM={gpu_memory_gib:.1f} GiB, "
    f"CPU={cpu_count}"
)

# ============================================================
# GPU-dependent configuration
# ============================================================

if "H100" in gpu_name:
    # H100 80GB / 94GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "256"
    VLLM_MAX_BATCHED_TOKENS = "65536"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    KDL_MAX_WORKERS = 64
    KDL_BBOX_MAX_WORKERS = 256
    KDL_RENDER_PROCESSES = min(32, cpu_count)

elif "A100" in gpu_name and gpu_memory_gib >= 70:
    # A100 80GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "128"
    VLLM_MAX_BATCHED_TOKENS = "32768"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    KDL_MAX_WORKERS = 48
    KDL_BBOX_MAX_WORKERS = 128
    KDL_RENDER_PROCESSES = min(24, cpu_count)

elif "A100" in gpu_name:
    # A100 40GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "64"
    VLLM_MAX_BATCHED_TOKENS = "16384"
    VLLM_GPU_MEMORY_UTILIZATION = "0.95"

    KDL_MAX_WORKERS = 32
    KDL_BBOX_MAX_WORKERS = 64
    KDL_RENDER_PROCESSES = min(16, cpu_count)

elif "L4" in gpu_name:
    # NVIDIA L4 24GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "32"
    VLLM_MAX_BATCHED_TOKENS = "8192"
    VLLM_GPU_MEMORY_UTILIZATION = "0.92"

    KDL_MAX_WORKERS = 16
    KDL_BBOX_MAX_WORKERS = 32
    KDL_RENDER_PROCESSES = min(8, cpu_count)

elif "T4" in gpu_name:
    # NVIDIA T4 16GB
    VLLM_DTYPE = "float16"
    VLLM_MAX_NUM_SEQS = "8"
    VLLM_MAX_BATCHED_TOKENS = "4096"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    KDL_MAX_WORKERS = 8
    KDL_BBOX_MAX_WORKERS = 16
    KDL_RENDER_PROCESSES = min(4, cpu_count)

else:
    # Safe fallback
    VLLM_DTYPE = "auto"
    VLLM_MAX_NUM_SEQS = "8"
    VLLM_MAX_BATCHED_TOKENS = "4096"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    KDL_MAX_WORKERS = 8
    KDL_BBOX_MAX_WORKERS = 16
    KDL_RENDER_PROCESSES = min(4, cpu_count)


# ============================================================
# Export vLLM config to %%bash
# ============================================================

os.environ["VLLM_DTYPE"] = VLLM_DTYPE
os.environ["VLLM_MAX_NUM_SEQS"] = VLLM_MAX_NUM_SEQS
os.environ["VLLM_MAX_BATCHED_TOKENS"] = VLLM_MAX_BATCHED_TOKENS
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = VLLM_GPU_MEMORY_UTILIZATION


print("\nvLLM config:")
print(f"  dtype={VLLM_DTYPE}")
print(f"  max_num_seqs={VLLM_MAX_NUM_SEQS}")
print(f"  max_num_batched_tokens={VLLM_MAX_BATCHED_TOKENS}")
print(f"  gpu_memory_utilization={VLLM_GPU_MEMORY_UTILIZATION}")

print("\nKDL pipeline config:")
print(f"  max_workers={KDL_MAX_WORKERS}")
print(f"  bbox_max_workers={KDL_BBOX_MAX_WORKERS}")
print(f"  render_processes={KDL_RENDER_PROCESSES}")

GPU=NVIDIA L4, VRAM=22.0 GiB, CPU=12

vLLM config:
  dtype=bfloat16
  max_num_seqs=32
  max_num_batched_tokens=8192
  gpu_memory_utilization=0.92

KDL pipeline config:
  max_workers=16
  bbox_max_workers=32
  render_processes=8


In [6]:
%%bash

echo "Starting vLLM with:"
echo "  dtype=$VLLM_DTYPE"
echo "  max_num_seqs=$VLLM_MAX_NUM_SEQS"
echo "  max_num_batched_tokens=$VLLM_MAX_BATCHED_TOKENS"
echo "  gpu_memory_utilization=$VLLM_GPU_MEMORY_UTILIZATION"

nohup vllm serve KDLAI/KDL-Frontier-Parser-nano \
    --host 0.0.0.0 \
    --port 8000 \
    --served-model-name kdl-frontier-parser-nano \
    --dtype "$VLLM_DTYPE" \
    --max-model-len 8192 \
    --max-num-seqs "$VLLM_MAX_NUM_SEQS" \
    --max-num-batched-tokens "$VLLM_MAX_BATCHED_TOKENS" \
    --gpu-memory-utilization "$VLLM_GPU_MEMORY_UTILIZATION" \
    --limit-mm-per-prompt '{"image":1}' \
    --trust-remote-code \
    --enable-chunked-prefill \
    --enable-prefix-caching \
    --generation-config vllm \
    > /content/vllm-kdl.log 2>&1 &

echo $! > /content/vllm.pid

echo "vLLM PID: $(cat /content/vllm.pid)"
echo "Log: /content/vllm-kdl.log"

Starting vLLM with:
  dtype=bfloat16
  max_num_seqs=32
  max_num_batched_tokens=8192
  gpu_memory_utilization=0.92
vLLM PID: 4428
Log: /content/vllm-kdl.log


In [7]:
!nvidia-smi

Tue Aug 11 07:17:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# import os
# import signal
# import time

# try:
#     vllm_pid = int(
#         subprocess.check_output(
#             "pgrep -f 'vllm serve'",
#             shell=True
#         ).decode().strip()
#     )

#     print(f"Stopping vLLM PID={vllm_pid}")

#     os.kill(vllm_pid, signal.SIGTERM)

#     time.sleep(10)

#     print("Server stopped.")

# except Exception as e:
#     print("Không tìm thấy vLLM process:", e)

In [9]:
import requests
import time

HEADERS = {
    "Authorization": f"Bearer {VLLM_API_KEY}",
}

deadline = time.time() + 30 * 60
ready = False

while time.time() < deadline:

    try:
        response = requests.get(
            "http://127.0.0.1:8000/v1/models",
            headers=HEADERS,
            timeout=5,
        )

        if response.status_code == 200:
            print(response.json())
            ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(10)

if not ready:
    # In log khi fail
    print("--- vLLM log ---")
    !tail -200 vllm.log
    raise RuntimeError("vLLM chưa sẵn sàng.")

print("Chandra vLLM ready.")

{'object': 'list', 'data': [{'id': 'kdl-frontier-parser-nano', 'object': 'model', 'created': 1786432784, 'owned_by': 'vllm', 'root': 'KDLAI/KDL-Frontier-Parser-nano', 'parent': None, 'max_model_len': 8192, 'permission': [{'id': 'modelperm-8499450971d669e8', 'object': 'model_permission', 'created': 1786432784, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}
Chandra vLLM ready.


In [10]:
# from pyngrok import ngrok

# ngrok.set_auth_token(NGROK_AUTHTOKEN)

# # Đóng tunnel cũ nếu chạy lại cell.
# ngrok.kill()

# tunnel = ngrok.connect(
#     addr=8000,
#     proto="http",
#     bind_tls=True,
# )

# PUBLIC_URL = tunnel.public_url.rstrip("/")

# print("Public URL:", PUBLIC_URL)
# print("VLLM_API_BASE:", f"{PUBLIC_URL}/v1")


# response = requests.get(
#     f"{PUBLIC_URL}/v1/models",
#     headers=HEADERS,
#     timeout=30,
# )

# print(response.status_code)
# print(response.json())

In [11]:
%%bash
git clone -b baseline --single-branch \
  https://github.com/iSE-UET-VNU/AXIOM_DE-RD.git \
  /content/AXIOM_DE-RD

Cloning into '/content/AXIOM_DE-RD'...


In [12]:
%cd /content/AXIOM_DE-RD
!pip install -e .
!python research/experiments/fetch_pdfs.py physics /content/vidore_v3_physics


/content/AXIOM_DE-RD
Obtaining file:///content/AXIOM_DE-RD
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 109.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 158.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 139.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 150.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2

In [13]:
import os
import requests

VLLM_API_BASE = "http://127.0.0.1:8000/v1"

import getpass
VLLM_API_KEY = getpass.getpass("VLLM_API_KEY: ")

os.environ["VLLM_API_BASE"] = VLLM_API_BASE
os.environ["VLLM_API_KEY"] = VLLM_API_KEY

headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}

models_response = requests.get(
    f"{VLLM_API_BASE}/models",
    headers=headers,
    timeout=30,
)
models_response.raise_for_status()

models = models_response.json()["data"]
model_name = models[0]["id"]
os.environ["VLLM_MODEL_NAME"] = model_name

openapi_response = requests.get(
    "http://127.0.0.1:8000/openapi.json",
    timeout=30,
)
openapi_response.raise_for_status()
paths = openapi_response.json().get("paths", {})

assert "/v1/chat/completions/batch" in paths, (
    "vLLM server hiện tại không có /v1/chat/completions/batch"
)

print("vLLM model:", model_name)
print("Batch endpoint: OK")
print("Colab CPU count:", os.cpu_count())

vLLM model: kdl-frontier-parser-nano
Batch endpoint: OK
Colab CPU count: 12


In [15]:
!apt-get -qq install -y poppler-utils
!python /content/AXIOM_DE-RD/research/experiments/fetch_pdfs.py physics /content/vidore_v3_physics


E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/p/poppler/poppler-utils_22.02.0-2ubuntu0.12_amd64.deb  404  Not Found [IP: 91.189.91.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
  [1/42] Autrement_Ch-0-Presentation_cours_autrement.pdf
  [2/42] Autrement_Ch-1a-La-quete-de-lunite-La-methode-scientifique.pdf
  [3/42] Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023.pdf
  [4/42] Autrement_Ch-2-Des-particules-aux-systemes.pdf
  [5/42] Autrement_Ch-3a-La-complexite-les-sytemes-et-leurs-modeles.pdf
  [6/42] Autrement_Ch-3b-La-complexite-Science-des-systemes-complexes.pdf
  [7/42] Autrement_Ch-4a-Les-systemes-dynamiques.pdf
  [8/42] Autrement_Ch-4b-Le-Chaos-deterministe.pdf
  [9/42] Autrement_Ch-5-Le-Hasard-24.pdf
  [10/42] Autrement_Ch-6a-Modeles-bioinspires.pdf
  [11/42] Autrement_Ch-6b-Les-reseaux-dautomates.pdf
  [12/42] Autrement_Ch-7-Ordre-entropie-et-morphogenese.pdf
  [13/42] Cours_avance_Ch-0-Presentation_Cours_Avance.pdf
 

In [16]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [18]:
from pathlib import Path
import json, os

REPO_DIR = Path("/content/AXIOM_DE-RD")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/AXIOM_DE-RD/data")
except Exception as exc:
    print("Drive unavailable, writing to local disk:", exc)
    DATA_ROOT = Path("/content/data")

DATASET_DIR = Path("/content/vidore_v3_physics")
assert DATASET_DIR.is_dir(), f"Không tìm thấy dataset: {DATASET_DIR}"


RUN_NAME = "vidore-v3-physics-kdl"

import getpass
# Lấy key từ Colab Secrets.
openrouter_api_key = getpass.getpass("OPENROUTER_API_KEY: ")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

source_config = (
    REPO_DIR / "configs/pipeline.vidore-v3-kdl.yaml"
)

config = json.loads(source_config.read_text(encoding="utf-8"))

# Input
config["local_input"]["path"] = str(DATASET_DIR)
config["local_input"]["recursive"] = True
config["local_input"]["include_extensions"] = [".pdf"]

# Bật toàn bộ pipeline
config["enabled_modules"] = [
    "ingestion",
    "cleaning",
    "enrichment",
    "chunking_embedding",
    "integration",
    "artifacts",
]

# Giữ từng stage trong data/<stage>/benchmarks/...
config["ingested_dir"] = str(
    DATA_ROOT / f"ingested/benchmarks/{RUN_NAME}"
)
config["cleaned_dir"] = str(
    DATA_ROOT / f"cleaned/benchmarks/{RUN_NAME}"
)
config["enriched_dir"] = str(
    DATA_ROOT / f"enriched/benchmarks/{RUN_NAME}"
)
config["embedded_dir"] = str(
    DATA_ROOT / f"embedded/benchmarks/{RUN_NAME}"
)
config["output_dir"] = str(
    DATA_ROOT / f"output/benchmarks/{RUN_NAME}"
)
config["parsing"]["kdl"]["output_dir"] = str(
    DATA_ROOT / f"work/benchmarks/{RUN_NAME}"
)

# KDL endpoint/model
kdl_config = config["parsing"]["kdl"]

kdl_config["endpoint_url"] = (
    "http://127.0.0.1:8000/v1"
)

kdl_config["model"] = (
    "kdl-frontier-parser-nano"
)


config["chunking_embedding"] = {
    "chunker": "fixed_overlap",
    "chunker_params": {
        "n_words": 512,
        "overlap": 128,
    },
    "max_rows_per_chunk": 20,
    "embedder": "openrouter_te3s",
    "embedder_params": {
        "model": "openai/text-embedding-3-small",
        "dimension": 1536,
        "api_key_env": "OPENROUTER_API_KEY",
        "batch_size": 64,
        "cache_dir": str(
            DATA_ROOT / "work/embedding_cache/text-embedding-3-small"
        ),
        "app_title": "AXIOM_DE-RD",
    },
    "retrieval_profile": "hybrid_default",
}


RUNTIME_CONFIG = Path(f"/content/pipeline.{RUN_NAME}-full.json")


RUNTIME_CONFIG.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print("Runtime config:", RUNTIME_CONFIG)
print("Dataset:", config["local_input"]["path"])
print("Modules:", config["enabled_modules"])
print("Output:", config["output_dir"])
print("KDL endpoint:", kdl_config["endpoint_url"])
print("KDL model:", kdl_config["model"])
print(
    "Embedding model:",
    config["chunking_embedding"]
    ["embedder_params"]["model"],
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime config: /content/pipeline.vidore-v3-physics-kdl-full.json
Dataset: /content/vidore_v3_physics
Modules: ['ingestion', 'cleaning', 'enrichment', 'chunking_embedding', 'integration', 'artifacts']
Output: /content/drive/MyDrive/AXIOM_DE-RD/data/output/benchmarks/vidore-v3-physics-kdl
KDL endpoint: http://127.0.0.1:8000/v1
KDL model: kdl-frontier-parser-nano
Embedding model: openai/text-embedding-3-small


In [19]:
import json
from pathlib import Path

RUNTIME_CONFIG = Path("/content/pipeline.vidore-v3-physics-kdl-full.json")


config = json.loads(
    RUNTIME_CONFIG.read_text(encoding="utf-8")
)

kdl_config = config["parsing"]["kdl"]

kdl_config["continuous_page_queue"] = True

# ============================================================
# Worker configuration based on detected GPU/CPU profile
# ============================================================

# Số page pipeline đồng thời.
# Đồng thời là số ảnh trang tối đa nằm trong RAM.
kdl_config["max_workers"] = KDL_MAX_WORKERS

# Số process render PDF.
kdl_config["render_processes"] = KDL_RENDER_PROCESSES

# Tổng số request nhận dạng bbox đồng thời.
kdl_config["bbox_max_workers"] = KDL_BBOX_MAX_WORKERS


# ============================================================
# Request configuration
# ============================================================

kdl_config["request_timeout_seconds"] = 3600
kdl_config["max_retries"] = 2


# ============================================================
# Token budgets đúng ParseBench
# ============================================================

kdl_config["layout_max_output_tokens"] = 6000
kdl_config["text_max_output_tokens"] = 2048
kdl_config["table_max_output_tokens"] = 5500
kdl_config["picture_max_output_tokens"] = 4096
kdl_config["formula_max_output_tokens"] = 128


# ============================================================
# Save config
# ============================================================

RUNTIME_CONFIG.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)


# ============================================================
# Print applied configuration
# ============================================================

print({
    "continuous_page_queue":
        kdl_config["continuous_page_queue"],

    "max_workers":
        kdl_config["max_workers"],

    "render_processes":
        kdl_config["render_processes"],

    "bbox_max_workers":
        kdl_config["bbox_max_workers"],

    "request_timeout_seconds":
        kdl_config["request_timeout_seconds"],

    "max_retries":
        kdl_config["max_retries"],
})

{'continuous_page_queue': True, 'max_workers': 16, 'render_processes': 8, 'bbox_max_workers': 32, 'request_timeout_seconds': 3600, 'max_retries': 2}


In [20]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/AXIOM_DE-RD")

RUNTIME_CONFIG = Path("/content/pipeline.vidore-v3-physics-kdl-full.json")


LOG_PATH = Path("/content/vidore-v3-physics-kdl.log")


log_handle = LOG_PATH.open(
    "w",
    encoding="utf-8",
)

pipeline_process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "scripts/run_pipeline.py",
        "--config",
        str(RUNTIME_CONFIG),
    ],
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)

log_handle.close()

print("Pipeline PID:", pipeline_process.pid)
print("Pipeline log:", LOG_PATH)
print("vLLM log: /content/vllm-kdl.log")

Pipeline PID: 7340
Pipeline log: /content/vidore-v3-physics-kdl.log
vLLM log: /content/vllm-kdl.log


In [21]:
# pipeline_process.terminate()
# pipeline_process.wait(timeout=30)
# log_handle.close()

# print("Pipeline đã dừng")

In [24]:
!tail -20 /content/vidore-v3-physics-kdl.log
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv


INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://127.0.0.1:8000/v1/chat/completions "HTTP/

In [47]:
!grep -c "chat/completions" /content/vidore-v3-physics-kdl.log
!ls /content/drive/MyDrive/AXIOM_DE-RD/data/ingested/benchmarks/vidore-v3-physics-kdl/*/documents/*.json 2>/dev/null | wc -l


23210
42


In [48]:
from pathlib import Path

INGESTED_ROOT = DATA_ROOT / f"ingested/benchmarks/{RUN_NAME}"
print("root exists:", INGESTED_ROOT.is_dir())

runs = sorted((p for p in INGESTED_ROOT.iterdir() if p.is_dir()),
              key=lambda p: p.stat().st_mtime)
latest = runs[-1]
docs = list((latest / "documents").glob("*.json"))
print("Run:", latest.name)
print("Persisted documents:", len(docs), "/ 42")
print("Metadata exists:", (latest / "metadata.json").is_file())


root exists: True
Run: 0183f2c78ac7802b
Persisted documents: 42 / 42
Metadata exists: True


In [52]:
import json
from pathlib import Path

cfg = json.loads(Path("/content/pipeline.vidore-v3-physics-kdl-full.json").read_text())
for stage in ("ingested_dir", "cleaned_dir", "enriched_dir", "output_dir"):
    root = Path(cfg[stage])
    runs = sorted((p for p in root.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime) if root.is_dir() else []
    n = len(list((runs[-1] / "documents").glob("*.json"))) if runs else 0
    print(f"{stage:14s} {n:3d}/42  {runs[-1].name if runs else '(none)'}")

!mountpoint -q /content/drive && echo "REAL Drive mount" || echo "NOT mounted — local disk only"


ingested_dir    42/42  0183f2c78ac7802b
cleaned_dir     42/42  0183f2c78ac7802b
enriched_dir    42/42  0183f2c78ac7802b
output_dir      42/42  0183f2c78ac7802b
REAL Drive mount


In [53]:
!cd /content/drive/MyDrive/AXIOM_DE-RD/data && zip -qr /content/physics-kdl.zip \
    output/benchmarks/vidore-v3-physics-kdl ingested/benchmarks/vidore-v3-physics-kdl
!du -h /content/physics-kdl.zip
from google.colab import files; files.download("/content/physics-kdl.zip")


50M	/content/physics-kdl.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>